# Dataset Generation

## Objective

This notebook is responsible for generating and preparing the datasets used throughout the Marketing Attribution & ROI Dashboard project.

The datasets include:

- Ad Spend Data
- Web Analytics Log
- CRM Conversion Data

These datasets simulate real-world marketing campaign performance and customer conversion behavior.

In [19]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

np.random.seed(42)
random.seed(42)

NUM_USERS = 2000
NUM_DAYS = 90
START_DATE = datetime(2024, 1, 1)

CHANNELS = ['Google_Search', 'Meta_Facebook', 'TikTok', 'LinkedIn', 'Email', 'Organic']
CAMPAIGNS = {
    'Google_Search': ['GSearch_Brand', 'GSearch_Generic', 'GSearch_Competitor'],
    'Meta_Facebook': ['FB_Retargeting', 'FB_Lookalike', 'FB_Awareness'],
    'TikTok': ['TikTok_Video1', 'TikTok_Promo'],
    'LinkedIn': ['LinkedIn_B2B', 'LinkedIn_Lead'],
    'Email': ['Email_Welcome', 'Email_Promo', 'Email_Retention'],
    'Organic': ['Organic_SEO', 'Organic_Direct']
}
UTM_SOURCES = {
    'Google_Search': 'google',
    'Meta_Facebook': 'facebook',
    'TikTok': 'tiktok',
    'LinkedIn': 'linkedin',
    'Email': 'email',
    'Organic': 'organic'
}
PAGES = ['/home', '/product', '/pricing', '/about', '/blog', '/checkout', '/signup']

In [20]:
# FILE 1 - Ad Spend
print("Generating Ad Spend Data...")
ad_spend_rows = []
for day_offset in range(NUM_DAYS):
    date = START_DATE + timedelta(days=day_offset)
    for channel in CHANNELS:
        for campaign in CAMPAIGNS[channel]:
            spend_range = {
                'Google_Search': (500, 3000),
                'Meta_Facebook': (300, 2000),
                'TikTok': (200, 1500),
                'LinkedIn': (400, 2500),
                'Email': (50, 300),
                'Organic': (0, 0)
            }
            low, high = spend_range[channel]
            spend = round(random.uniform(low, high), 2)
            clicks = int(spend / random.uniform(0.5, 3.0)) if spend > 0 else random.randint(50, 500)
            impressions = clicks * random.randint(10, 50)
            ad_spend_rows.append({
                'date': date.strftime('%Y-%m-%d'),
                'channel': channel,
                'campaign': campaign,
                'amount_spent_usd': spend,
                'clicks': clicks,
                'impressions': impressions,
                'cpc': round(spend / clicks, 3) if clicks > 0 else 0
            })
ad_spend_df = pd.DataFrame(ad_spend_rows)
ad_spend_df.to_csv('ad_spend_data.csv', index=False)
print(f"✅ ad_spend_data.csv created — {len(ad_spend_df)} rows")

Generating Ad Spend Data...
✅ ad_spend_data.csv created — 1350 rows


In [21]:
# FILE 2 - Web Analytics
print("Generating Web Analytics Log...")
web_log_rows = []
user_ids = [f'USR_{str(i).zfill(5)}' for i in range(1, NUM_USERS + 1)]
for user_id in user_ids:
    num_touchpoints = random.randint(1, 5)
    journey_start = START_DATE + timedelta(days=random.randint(0, NUM_DAYS - 10))
    for tp in range(num_touchpoints):
        session_date = journey_start + timedelta(days=random.randint(0, 7))
        channel = random.choices(CHANNELS, weights=[30, 25, 15, 10, 10, 10])[0]
        campaign = random.choice(CAMPAIGNS[channel])
        utm_source = UTM_SOURCES[channel] if random.random() > 0.1 else None
        utm_medium = 'cpc' if channel in ['Google_Search', 'Meta_Facebook', 'TikTok', 'LinkedIn'] else 'email' if channel == 'Email' else 'organic'
        utm_campaign = campaign if random.random() > 0.05 else None
        web_log_rows.append({
            'session_id': f'SES_{random.randint(100000, 999999)}',
            'user_id': user_id,
            'timestamp': session_date.strftime('%Y-%m-%d') + f' {random.randint(0,23):02d}:{random.randint(0,59):02d}:00',
            'channel': channel,
            'utm_source': utm_source,
            'utm_medium': utm_medium,
            'utm_campaign': utm_campaign,
            'page_visited': random.choice(PAGES),
            'session_duration_sec': random.randint(30, 900),
            'device': random.choice(['mobile', 'desktop', 'tablet'])
        })
web_log_df = pd.DataFrame(web_log_rows)
web_log_df.to_csv('web_analytics_log.csv', index=False)
print(f"✅ web_analytics_log.csv created — {len(web_log_df)} rows")

Generating Web Analytics Log...
✅ web_analytics_log.csv created — 6034 rows


In [22]:
# FILE 3 - CRM Only
print("Generating CRM Conversion Data...")
crm_rows = []
converting_users = random.sample(user_ids, int(NUM_USERS * 0.30))
for user_id in converting_users:
    user_sessions = web_log_df[web_log_df['user_id'] == user_id]
    if user_sessions.empty:
        continue
    last_session = user_sessions.sort_values('timestamp').iloc[-1]
    conversion_date = pd.to_datetime(last_session['timestamp']) + timedelta(days=random.randint(0, 2))
    revenue = round(random.uniform(20, 500), 2)
    last_channel = last_session['channel']
    crm_rows.append({
        'customer_id': user_id,
        'conversion_date': conversion_date.strftime('%Y-%m-%d'),
        'revenue_usd': revenue,
        'last_touch_channel': last_channel,
        'last_touch_campaign': last_session['utm_campaign'],
        'product_purchased': random.choice(['Basic Plan', 'Pro Plan', 'Enterprise Plan', 'One-Time Purchase']),
        'country': random.choice(['India', 'USA', 'UK', 'Germany', 'Australia', 'Canada'])
    })
crm_df = pd.DataFrame(crm_rows)
crm_df.to_csv('crm_conversion_data.csv', index=False)
print(f"✅ crm_conversion_data.csv created — {len(crm_df)} rows")
print("\n🎉 DONE! Teeno CSV files ban gayi!")

Generating CRM Conversion Data...
✅ crm_conversion_data.csv created — 600 rows

🎉 DONE! Teeno CSV files ban gayi!


In [23]:
import os
files = ['ad_spend_data.csv', 'web_analytics_log.csv', 'crm_conversion_data.csv']
for f in files:
    if os.path.exists(f):
        df = pd.read_csv(f)
        print(f"✅ {f} — {df.shape[0]} rows, {df.shape[1]} columns")
        

✅ ad_spend_data.csv — 1350 rows, 7 columns
✅ web_analytics_log.csv — 6034 rows, 10 columns
✅ crm_conversion_data.csv — 600 rows, 7 columns


## Conclusion

The generated datasets will be used in subsequent notebooks for data cleaning, exploratory analysis, KPI calculations, SQL integration, attribution modeling, and dashboard creation.